In [91]:
import os
import sys
import yaml
import pandas as pd
import numpy as np

with open('../../config.local.yaml', 'r') as f:
    local_config = yaml.safe_load(f)

LOCAL_PATH = local_config['LOCAL_PATH']

sys.path.append(os.path.join(LOCAL_PATH, "src/python"))

import writing_tools as wt
from utils import stars_p

from matplotlib import pyplot as plt
from scipy.stats import f_oneway

plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['font.size'] = 11

os.environ['LOKY_MAX_CPU_COUNT'] = '1' # because of windows core count warning

with open('../../config.local.yaml', 'r') as f:
    local_config = yaml.safe_load(f)
with open('../../config.yaml', 'r') as f:
    config = yaml.safe_load(f)

LOCAL_PATH = local_config['LOCAL_PATH']
DATA_PATH = local_config['DATA_PATH']
EMBEDDING_DIMENSION = config['EMBEDDING_DIMENSION']

rng = np.random.default_rng(12898)

N_CLUSTERS = 3
N_COMPONENTS = 10

In [92]:
df = pd.read_parquet(os.path.join(DATA_PATH, "intermediate_data/cpc", "analysis_data_w_embeddings.parquet"))
reasons_df = pd.read_parquet(os.path.join(DATA_PATH, "intermediate_data/cpc", "reasons.parquet"))


In [93]:
reasons_agg_df = reasons_df.groupby('reason').agg(count=('item_no','count')).sort_values(by='count', ascending=False).reset_index()

In [94]:
df['outcome'] = df['project_result'].map({
    'APPROVED': 'Approved',
    'APPROVED WITH MINOR CONDITIONS OR MODIFICATIONS': 'Approved w/ conditions',
    'APPROVED WITH MAJOR CONDITIONS OR MODIFICATIONS': 'Approved w/ conditions',
    'DENIED': 'Delayed / Denied',
    'APPLICATION WITHDRAWN': 'Delayed / Denied',
    'DELAYED': 'Delayed / Denied'
})
n_approved = f"{(df['outcome'] == "Approved").sum():,.0f}"
n_approved_w_conditions = f"{(df['outcome'] == "Approved w/ conditions").sum():,.0f}"
n_delayed_denied = f"{(df['outcome'] == "Delayed / Denied").sum():,.0f}"

In [95]:
def get_descriptive_stats_continuous(variable):
    # Get descriptive stats for continuous variables
    approved_mask = df['outcome'] == "Approved"
    approved_w_conditions_mask = df['outcome'] == "Approved w/ conditions"
    delayed_denied_mask = df['outcome'] == "Delayed / Denied"
    
    # Calculate mean and standard deviation for approved cases
    approved_mean = df.loc[approved_mask, variable].mean()
    approved_std = df.loc[approved_mask, variable].std()
    
    # Calculate mean and standard deviation for approved w/ conditions cases
    approved_w_conditions_mean = df.loc[approved_w_conditions_mask, variable].mean()
    approved_w_conditions_std = df.loc[approved_w_conditions_mask, variable].std()

    # Calculate mean and standard deviation for delayed/denied cases
    delayed_denied_mean = df.loc[delayed_denied_mask, variable].mean()
    delayed_denied_std = df.loc[delayed_denied_mask, variable].std()
    
    # Calculate F-test for mean differences between groups
    f_stat, p_value = f_oneway(
        df.loc[approved_mask, variable],
        df.loc[approved_w_conditions_mask, variable],
        df.loc[delayed_denied_mask, variable]
    )
    
    return approved_mean, approved_w_conditions_mean, delayed_denied_mean, f_stat, p_value

In [96]:
header = fr"""
\begin{{table}}[H]
\centering
\caption{{Descriptive Statistics by Motion Outcome--Public Input}}
\vspace{{0.2cm}}
\label{{tab_bivariate_descriptives_public_input}}
\begin{{adjustbox}}{{max width=\textwidth}}
\begin{{threeparttable}}
\begin{{tabular}}{{lrrrll}} \toprule
 Variable & \makecell{{Approved \\ ($n = {n_approved}$)}} & \makecell{{Approved w/ \\ conditions \\ ($n = {n_approved_w_conditions}$)}} & \makecell{{Delayed / \\ denied \\ ($n = {n_delayed_denied}$)}} & \makecell{{Test stat}} & \makecell{{p-value}}  \\ \midrule
 & & & & & \\
"""
footer = r"""
 & & & & & \\
\bottomrule
\end{tabular}
\begin{tablenotes}
\item {\textit{Notes: } This reports descriptive statistics for variables grouped by motion outcome. To test differences between groups, a one-way ANOVA F-test was conducted for continuous variables and a chi-square test was conducted for categorical variables.}
\end{tablenotes}
\end{threeparttable}
\end{adjustbox}
\end{table}
"""
tbl = ""

var = "n_docs"
varname = "Number of letters (mean)"
results = get_descriptive_stats_continuous(var)
tbl += rf"{varname} & {results[0]:.2f} & {results[1]:.2f} & {results[2]:.2f} & $F = {results[3]:.2f}$ & ${results[4]:.3f}^{{{stars_p(results[4])}}}$ \\" + "\n"

var = "n_support"
varname = "~ ~ in support"
results = get_descriptive_stats_continuous(var)
tbl += rf"{varname} & {results[0]:.2f} & {results[1]:.2f} & {results[2]:.2f} & $F = {results[3]:.2f}$ & ${results[4]:.3f}^{{{stars_p(results[4])}}}$ \\" + "\n"

tbl += r"~ ~ ~ mentioning \ldots & & & & & \\" + "\n"
for r in reasons_agg_df['reason']:
    mask = (reasons_df['reason']==r) & (reasons_df['support_or_oppose'].isin(['DEFINITELY SUPPORT', 'SOMEWHAT SUPPORT']))
    temp_df = reasons_df.loc[mask].groupby(['date', 'year', 'item_no']).agg(temp = ('reason','count')).reset_index()
    df = df.merge(temp_df, on=['date', 'year', 'item_no'], how='left')
    df['temp'] = df['temp'].fillna(0)
    df['pct_temp'] = (df['temp'] / df['n_docs'].clip(lower=1))*100
    var = "temp"
    varname = f"~ ~ ~ ~ {r}"
    results = get_descriptive_stats_continuous(var)
    #tbl += rf"{varname} & {results[0]:.2f}\% & {results[1]:.2f}\% & {results[2]:.2f}\% & $F = {results[3]:.2f}$ & ${results[4]:.3f}^{{{stars_p(results[4])}}}$ \\" + "\n"
    tbl += rf"{varname} & {results[0]:.2f} & {results[1]:.2f} & {results[2]:.2f} & $F = {results[3]:.2f}$ & ${results[4]:.3f}^{{{stars_p(results[4])}}}$ \\" + "\n"
    df = df.drop(columns=['temp', 'pct_temp'])

var = "n_oppose"
varname = "~ ~ in opposition"
results = get_descriptive_stats_continuous(var)
tbl += rf"{varname} & {results[0]:.2f} & {results[1]:.2f} & {results[2]:.2f} & $F = {results[3]:.2f}$ & ${results[4]:.3f}^{{{stars_p(results[4])}}}$ \\" + "\n"

tbl += r"~ ~ ~ mentioning \ldots & & & & & \\" + "\n"
for r in reasons_agg_df['reason']:
    mask = (reasons_df['reason']==r) & (reasons_df['support_or_oppose'].isin(['DEFINITELY OPPOSE', 'SOMEWHAT OPPOSE']))
    temp_df = reasons_df.loc[mask].groupby(['date', 'year', 'item_no']).agg(temp = ('reason','count')).reset_index()
    df = df.merge(temp_df, on=['date', 'year', 'item_no'], how='left')
    df['temp'] = df['temp'].fillna(0)
    df['pct_temp'] = (df['temp'] / df['n_docs'].clip(lower=1))*100
    var = "temp"
    varname = f"~ ~ ~ ~ {r}"
    results = get_descriptive_stats_continuous(var)
    #tbl += rf"{varname} & {results[0]:.2f}\% & {results[1]:.2f}\% & {results[2]:.2f}\% & $F = {results[3]:.2f}$ & ${results[4]:.3f}^{{{stars_p(results[4])}}}$ \\" + "\n"
    tbl += rf"{varname} & {results[0]:.2f} & {results[1]:.2f} & {results[2]:.2f} & $F = {results[3]:.2f}$ & ${results[4]:.3f}^{{{stars_p(results[4])}}}$ \\" + "\n"
    df = df.drop(columns=['temp', 'pct_temp'])



print(header + tbl + footer)

with open(os.path.join(LOCAL_PATH, "tables", "tab_bivariate_descriptives_public_input.tex"), "w", encoding='utf-8') as f:
    f.write(header + tbl + footer)




\begin{table}[H]
\centering
\caption{Descriptive Statistics by Motion Outcome--Public Input}
\vspace{0.2cm}
\label{tab_bivariate_descriptives_public_input}
\begin{adjustbox}{max width=\textwidth}
\begin{threeparttable}
\begin{tabular}{lrrrll} \toprule
 Variable & \makecell{Approved \\ ($n = 354$)} & \makecell{Approved w/ \\ conditions \\ ($n = 328$)} & \makecell{Delayed / \\ denied \\ ($n = 136$)} & \makecell{Test stat} & \makecell{p-value}  \\ \midrule
 & & & & & \\
Number of letters (mean) & 4.41 & 10.04 & 11.12 & $F = 6.45$ & $0.002^{***}$ \\
~ ~ in support & 1.94 & 4.67 & 3.26 & $F = 3.64$ & $0.027^{**}$ \\
~ ~ ~ mentioning \ldots & & & & & \\
~ ~ ~ ~ housing supply / affordability & 1.41 & 2.99 & 1.07 & $F = 2.30$ & $0.101^{}$ \\
~ ~ ~ ~ neighborhood character & 0.42 & 1.24 & 0.76 & $F = 5.80$ & $0.003^{***}$ \\
~ ~ ~ ~ procedural issues & 0.50 & 1.68 & 1.04 & $F = 2.80$ & $0.062^{*}$ \\
~ ~ ~ ~ building height / size / density & 0.32 & 1.23 & 0.65 & $F = 1.16$ & $0.313^{}$ \\
~ 